# Curation of GP clinical dataset

- Cleaning the event dates
- Makign a curated code column from READ2 and CTV3 columns

In [ ]:
import os
import sys

root_path = os.path.dirname(os.path.abspath(os.path.dirname('__file__')))
sys.path.insert(0, root_path)

In [ ]:
from util.general_utils import field_availability_check, list_field_instances
from util.dask_utils import import_field_from_dask
from util.parquet_maker import dask_to_parquet
from env.parameters import P
from util.logging import logging_setup

In [ ]:
import pandas as pd
import numpy as np
import  dask.dataframe as dd
pd.set_option('display.max_rows', 500)

In [ ]:
# load the cohort
df_cohort = pd.read_csv(f'''{P.output_csv_path}/cohort_1_base.csv''')
df_cohort.head()

In [ ]:
dd_cohort = dd.from_pandas(df_cohort, npartitions=2)
dd_cohort.head()

# GP data

In [ ]:
path_gp_clinical = f'''{P.output_parquet_path}/gp_clinical'''
dd_gp = dd.read_parquet(path_gp_clinical)
dd_gp.head()

In [ ]:
dd_gp["data_provider"].value_counts().compute()

In [ ]:
print(dd_gp["read_2"].isna().sum().compute())
print(dd_gp["read_3"].isna().sum().compute())

In [ ]:
print((dd_gp['read_3'].isnull() & (~dd_gp['read_2'].isnull())).sum().compute())


In [ ]:
print((~dd_gp['read_3'].isnull() & (dd_gp['read_2'].isnull())).sum().compute())


In [ ]:
print((~dd_gp['read_3'].isnull() & (~dd_gp['read_2'].isnull())).sum().compute())


In [ ]:

# there are no records with both values available

In [ ]:
dd_gp['read_curated'] = dd_gp['read_3'].fillna(dd_gp['read_2'])
dd_gp.head()


In [ ]:
#dd_gp['read_version'] = np.where((dd_gp['read_3'].notnull()), 'read_v3', 'read_v2')
dd_gp['read_version'] = dd_gp['read_3'].map_partitions(lambda df: df.notnull().astype(str).replace({'True': 'read_v3', 'False': 'read_v2'}), meta=('read_version', 'object'))

dd_gp.head()

In [ ]:
dd_gp.dtypes


In [ ]:
# type case

dd_gp['evdt_gp'] = dd.to_datetime(dd_gp['event_dt'], format= "%d/%m/%Y")
dd_gp.head()

In [ ]:
dd_gp["eid"] = dd_gp["eid"].astype("int64")

In [ ]:
dd_gp['evdt_gp'].isnull().sum().compute()


In [ ]:
dd_gp.head()

In [ ]:
dd_gp.dtypes

In [ ]:
dd_gp.shape[0].compute()

In [ ]:
# Clean
dd_gp_clean = dd_gp.copy()
dd_gp_clean= dd_gp_clean.dropna(subset=['evdt_gp'])


In [ ]:
dd_gp_clean = dd_gp_clean.dropna(subset=['read_curated'])


In [ ]:
#dd_gp_clean.shape[0].compute()


# Link with GP to filter based on dob and dod

In [ ]:
df_gp_j = dd.merge(dd_gp_clean, df_cohort[["eid", "dob", "dod", "evdt_ukb_end"]], on ="eid", how="inner")
df_gp_j.head()

In [ ]:
#df_gp_j.shape[0].compute()


In [ ]:
df_gp_j.dtypes

In [ ]:
# Change format and type
df_gp_j["evdt_gp"] = df_gp_j["evdt_gp"].dt.strftime("%Y-%m-%d")
df_gp_j["evdt_gp"] = dd.to_datetime(df_gp_j["evdt_gp"])

In [ ]:
df_gp_j['dob'] = dd.to_datetime(df_gp_j['dob'], format= "%Y-%m-%d")
df_gp_j['dod'] = dd.to_datetime(df_gp_j['dod'], format= "%Y-%m-%d")
df_gp_j['evdt_ukb_end'] = dd.to_datetime(df_gp_j['evdt_ukb_end'], format= "%Y-%m-%d")


In [ ]:
df_gp_j.head()

In [ ]:
df_gp_j.dtypes


In [ ]:
# how many before dob?
#print(df_gp_j[df_gp_j['evdt_gp']< df_gp_j['dob']].shape[0].compute())

In [ ]:
# how many after dod?
#print(df_gp_j[(df_gp_j['dod'].notnull())&(df_gp_j['evdt_gp']> df_gp_j['dod']) ].shape[0].compute())

In [ ]:
# how many after data end date?
#print(df_gp_j[df_gp_j['evdt_gp']> df_gp_j['data_end_date']].shape[0].compute())

In [ ]:
# drop these
df_final = df_gp_j.copy()
df_final = df_final[
    (df_final['evdt_gp']>=df_final['dob'])
    &
    ((df_final['dod'].isnull())|(df_final['dod']>=df_final['evdt_gp']))
    & (
        df_final['evdt_gp']<df_final['evdt_ukb_end']
    )
    ]

In [ ]:
#df_final.shape[0].compute()

In [ ]:
df_gp_sel = df_final[['eid', 'data_provider', 'read_curated', 'evdt_gp', 'value1', 'value2', 'value3']]
#df_gp_sel.head()

In [ ]:
#df_gp_sel.to_csv(f'''{P.output_csv_path}/df_gp_clean.csv''', index=False)


In [ ]:

dask_to_parquet(df_gp_sel, output_path=P.output_parquet_path, output_parquet_name="gp_clinical_curated", write_index=False)

In [ ]:
# read to check
df_test = dd.read_parquet(f'''{P.output_parquet_path}/gp_clinical_curated''')

In [ ]:
df_test.shape[0].compute()

In [ ]:
df_gp_j.shape[0].compute()
